# Transformer

Problem `gold-transformer` · gold

[GitHub notebook](https://github.com/ThomasKidane/portfolio2/blob/main/dl-guide/notebooks/gold/gold-transformer.ipynb) · self-reported local progress; no hosted judge.


In [ ]:
try:
    import dlguide
except ImportError:
    %pip -q install "dlguide[vision] @ git+https://github.com/ThomasKidane/portfolio2.git@24d60a86199b4b53ca417b2a1cf291932adfcb9a#subdirectory=dl-guide/dlguide"
from dlguide import Problem, Submission
P = Problem('gold-transformer', seed=0)
submission = None


## Task and constraints

Implement Transformer: derive numerically stable scaled dot-product attention and explicit mask semantics. Read the linked paper and isolate the named mechanism. Write a self-contained solve(data), fit using training tensors only, and return Submission(model=model). Keep validation labels out of fitting and preprocessing. This task isolates scaled dot-product attention, not the complete translation system. Boolean masks use True for allowed keys; additive masks are added before softmax; entirely masked rows must return zero. Use attention-equivalence; required output: forward(q,k,v,mask=None) -> attention output. Evaluate attention_max_error (minimize). Compare SDPA values and gradients including non-square and fully masked cases. Required ablation: Remove the inverse-square-root key-width scale and compare scores and gradient norms.

Metric: `{'name': 'attention_max_error', 'direction': 'minimize', 'threshold': None, 'atol': 1e-07, 'rtol': 1e-06, 'reference': 'torch.nn.functional.scaled_dot_product_attention', 'calibrated': False, 'pass_target': None, 'par_target': None, 'stretch_target': None, 'targets': {'pass': None, 'par': None, 'stretch': None}, 'target_status': 'provisional draft targets only; threshold remains null until calibration'}`

Budget: `{'wall_seconds': 60, 'parameters': 2000000, 'max_trainable_fraction': None, 'memory_mb': 1024, 'require_gpu': False, 'training_steps': 0, 'calibrated': False, 'note': 'Provisional safety caps, not empirically calibrated GPU budgets.'}`


## Refresher

Use train for fitting and val for choices. Keep held-out evaluation private by convention. Seed stochastic operations and compare a simple baseline first.


In [ ]:
data = None
try:
    data = P.data()
    print({name: len(data[name]) for name in ("train", "val")})
    print("Input and target shapes:", [tuple(t.shape) for t in data["train"].tensors])
except Exception as exc:
    P._failure = "Dataset unavailable: " + str(exc)
    print(P._failure)


In [ ]:
"""Implement scaled dot-product attention, not a full translation model."""
def solve(data):
    import math
    import torch
    from torch import nn
    from dlguide import Submission
    class Attention(nn.Module):
        def forward(self, q, k, v, mask=None):
            # q: [..., L, D], k: [..., S, D], v: [..., S, Dv]
            # TODO: scale dot products, apply True=allowed or additive mask,
            # normalize over keys, zero fully masked rows, multiply by v.
            raise NotImplementedError('Implement attention without calling SDPA')
    return Submission(model=Attention())


In [ ]:
try:
    if data is None:
        raise RuntimeError(P._failure or "Dataset unavailable")
    with P.budget() as budget:
        submission = solve(data)
        budget.validate(submission.model)
except NotImplementedError as exc:
    P._failure = "Starter incomplete: " + str(exc)
    submission = None
except Exception as exc:
    P._failure = type(exc).__name__ + ": " + str(exc)
    submission = None


In [ ]:
if submission is not None:
    try:
        P.check(submission)
    except Exception as exc:
        P._failure = "Self-check failed: " + str(exc)
else:
    P.check()


In [ ]:
verdict = P.grade(submission)


## Progressive hints

<details><summary>Hint 1</summary>

derive numerically stable scaled dot-product attention and explicit mask semantics

</details>

<details><summary>Hint 2</summary>

Write down every intermediate tensor shape before implementing the block.

</details>

<details><summary>Hint 3</summary>

Start with the smallest deterministic fixture and compare with a trusted primitive.

</details>

## Reflections

1. What baseline did you compare against?
2. What failed, and what evidence changed your next experiment?
3. How did validation guide choices without leaking held-out examples?
4. What would you change under half the time or parameter budget?

A local PASS is a learning signal, not a secure assessment. Only mark Solved when you understand your solution.
